 *Artificial Intelligence for Vision & NLP* &nbsp; | &nbsp;  *ATU Donegal - Postgrad Diploma in Big Data Analytics & Artificial Intelligence*

# Student Submission
Name           : Konrad Psiuk<br>
Student Number : L00169810<br>
Due Date       : 12th May 2026<br>
Assignment     : CA2 - Application<br>
Module         : AI for Vision and NLP<br>
Course         : Postgraduate Diploma in Big Data Analytics and AI


# 1. Setup

## 1.1 Install dependencies
Uncomment the cell below the first time you run the notebook.

In [ ]:
# %pip install --quiet spacy nltk scikit-learn pandas numpy matplotlib pillow pymupdf pytesseract opencv-python
# !python -m spacy download en_core_web_sm


## 1.2 Imports and global configuration

In [ ]:
# Core data / plotting
import os
import re
import json
from collections import Counter
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# OCR
import pytesseract
from pytesseract import Output

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# PDF -> image (used to make sample images reproducible)
import pymupdf

# NLTK resources (idempotent - safe to re-run)
for pkg in ('punkt', 'punkt_tab', 'stopwords', 'vader_lexicon'):
    nltk.download(pkg, quiet=True)

# spaCy model
nlp = spacy.load('en_core_web_sm')

# Tesseract binary path - on macOS Homebrew this is /opt/homebrew/bin/tesseract.
# Adjust if running on a different OS.
pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'

# Matplotlib default size
plt.rcParams['figure.figsize'] = (14, 8)

# Sample directory
SAMPLES_DIR = Path('samples')
assert SAMPLES_DIR.is_dir(), f'Missing samples directory at {SAMPLES_DIR.resolve()}'

ENGLISH_STOPWORDS = set(stopwords.words('english'))
PORTER = PorterStemmer()
SIA = SentimentIntensityAnalyzer()

print('cv2:', cv2.__version__, '| spaCy:', spacy.__version__, '| nltk:', nltk.__version__)
print('tesseract:', pytesseract.get_tesseract_version())


# 2. Sample documents

The `samples/` folder contains a mix of input types - **PDFs are fed in directly**, no manual conversion. The brief explicitly lists *scanned documents, photos of printed forms, PDFs, etc.* as valid inputs, so the loader below accepts any of them.

* `.jpg` / `.png` - read straight from disk with OpenCV.
* `.pdf` - opened with `pymupdf` and **every page** is rasterised at 200 dpi to a BGR array that the rest of the pipeline treats exactly like a scanned image. This is the same code path used by `Lab1c_WorkingWithPDFFiles` and `Lab_Tesseract_OCR`.


In [ ]:
MAX_PAGES_PER_PDF = 2  # cap pages from a single PDF so the report stays readable


def load_document_pages(path, max_pages=MAX_PAGES_PER_PDF):
    """Return [(label, bgr_image)] - one entry per page for PDFs, one per file otherwise."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}:
        return [(path.name, cv2.imread(str(path)))]
    if suffix == '.pdf':
        pages = []
        doc = pymupdf.open(str(path))
        try:
            for i, page in enumerate(doc):
                if max_pages is not None and i >= max_pages:
                    break
                pix = page.get_pixmap(dpi=200)
                arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
                if pix.n == 4:
                    bgr = cv2.cvtColor(arr, cv2.COLOR_RGBA2BGR)
                elif pix.n == 3:
                    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
                else:
                    bgr = cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
                pages.append((f'{path.name}#p{i+1}', bgr))
        finally:
            doc.close()
        return pages
    raise ValueError(f'Unsupported file type: {path}')


SUPPORTED = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.pdf'}
SAMPLE_FILES = sorted(p for p in SAMPLES_DIR.iterdir() if p.suffix.lower() in SUPPORTED)

# Expand into one entry per page so a multi-page PDF becomes multiple docs
SAMPLE_PAGES = []
for p in SAMPLE_FILES:
    for label, bgr in load_document_pages(p):
        SAMPLE_PAGES.append({'source': p, 'label': label, 'bgr': bgr})

for s in SAMPLE_PAGES:
    print(f"{s['label']:30s}  source={s['source'].name:25s}  shape={s['bgr'].shape}")


In [ ]:
n = len(SAMPLE_PAGES)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 8))
if n == 1:
    axes = [axes]
for ax, s in zip(axes, SAMPLE_PAGES):
    ax.imshow(cv2.cvtColor(s['bgr'], cv2.COLOR_BGR2RGB))
    ax.set_title(s['label'], fontsize=11)
    ax.axis('off')
plt.suptitle('Input documents (PDFs rendered on the fly)', fontsize=14)
plt.tight_layout()
plt.show()


# 3. Image preprocessing and enhancement

Tesseract's LSTM engine works best on a clean image, but the *best* preprocessing depends on the input - a clean digital page is hurt by an aggressive binary threshold that keeps printing artefacts, while a noisy scan benefits from one. The function below produces every intermediate stage and the OCR step that follows picks the variant that gives the highest mean confidence.

1. **Grayscale** to drop colour noise (Lab 6a).
2. **Median blur** to suppress salt-and-pepper artefacts (Lab 6a - *Median Blur on Noisy Image*).
3. **Image enhancement** - CLAHE (Contrast Limited Adaptive Histogram Equalisation) followed by an **unsharp mask**. CLAHE re-spreads the histogram locally so faint print becomes legible; the unsharp mask compensates for mild blur by subtracting a Gaussian-blurred copy of the image from itself. Together this is "contrast adjustment + deblurring" called out in the brief.
4. **Adaptive Gaussian threshold** (Lab 6a - *Adaptive Thresholding*) for binarisation - works better than global Otsu when lighting is uneven.
5. **Morphological opening** to remove isolated pixels that often trigger false OCR characters (Lab 6a - *Morphological Operations*).
6. **Conditional deskew** - estimate the skew angle with `cv2.minAreaRect` over the text pixels and only correct if the angle is in (0.5deg, 15deg). Outside that range the rotation step is skipped to avoid degrading already-aligned pages.

Each intermediate stage is returned so we can inspect what happens at every step and so the OCR layer can adaptively choose the best image.


In [ ]:
def estimate_skew_angle(binary):
    """Estimate skew in degrees from a 0/255 binary mask of dark pixels.

    `np.where` returns coordinates in (row, col) order. `cv2.minAreaRect`
    needs them in (x, y) order, so the columns are swapped before the call;
    forgetting this gives the rectangle of the *transposed* page and a
    nonsense ~90 degrees angle.
    """
    coords = np.column_stack(np.where(binary > 0))[:, [1, 0]]  # (x, y)
    if len(coords) < 100:
        return 0.0
    angle = cv2.minAreaRect(coords)[-1]
    # OpenCV returns the angle of the minor side relative to the x-axis.
    # Normalise so the value is in (-45, 45].
    if angle > 45:
        angle -= 90
    elif angle < -45:
        angle += 90
    # Page-level skews above 15 deg are almost always estimation errors on
    # documents with strong vertical/horizontal layouts, so ignore them.
    if abs(angle) > 15:
        return 0.0
    return float(angle)


def rotate_image(img, angle):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REPLICATE)


def enhance_image(gray):
    """Image refinement: CLAHE for local contrast + unsharp mask for
    sharpening (a simple deblurring technique)."""
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    contrast = clahe.apply(gray)
    blurred = cv2.GaussianBlur(contrast, (0, 0), sigmaX=3)
    sharpened = cv2.addWeighted(contrast, 1.5, blurred, -0.5, 0)
    return contrast, sharpened


def preprocess_image(img_bgr):
    """Return every preprocessing stage plus the estimated skew angle."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    denoised = cv2.medianBlur(gray, 3)
    contrast, sharpened = enhance_image(denoised)
    binary = cv2.adaptiveThreshold(
        sharpened, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, blockSize=31, C=15,
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)

    # Only rotate when there is a non-trivial skew - otherwise rotation
    # introduces interpolation blur that hurts already-aligned pages.
    angle = estimate_skew_angle(255 - cleaned)
    if abs(angle) >= 0.5:
        deskewed_gray = rotate_image(sharpened, angle)
        deskewed_bin = rotate_image(cleaned, angle)
    else:
        deskewed_gray = sharpened
        deskewed_bin = cleaned

    return {
        'gray': gray,
        'denoised': denoised,
        'contrast': contrast,
        'sharpened': sharpened,
        'binary': binary,
        'cleaned': cleaned,
        'deskewed_gray': deskewed_gray,
        'deskewed_bin': deskewed_bin,
        'angle': angle,
    }


In [ ]:
# Pick the densest page as the demo - it has the most structure to show off
demo_sample = max(SAMPLE_PAGES, key=lambda s: s['bgr'].shape[0] * s['bgr'].shape[1])
demo_label = demo_sample['label']
demo_img = demo_sample['bgr']
stages = preprocess_image(demo_img)
print(f'Demo page: {demo_label}')
print(f'Deskew angle: {stages["angle"]:.2f} deg')

fig, axes = plt.subplots(2, 3, figsize=(18, 14))
keys_titles = [
    ('gray',          '1. Grayscale'),
    ('denoised',      '2. Median-blur denoised'),
    ('contrast',      '3. CLAHE contrast enhancement'),
    ('sharpened',     '4. Unsharp-mask sharpening'),
    ('binary',        '5. Adaptive threshold (binarisation)'),
    ('deskewed_bin',  '6. Morph clean + conditional deskew'),
]
for ax, (key, title) in zip(axes.flat, keys_titles):
    ax.imshow(stages[key], cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle(f'Preprocessing + enhancement pipeline on {demo_label}', fontsize=14)
plt.tight_layout()
plt.show()


# 4. OCR text extraction

Tesseract is run twice and the output of the variant with the higher mean confidence is kept:

* once on the **raw grayscale** (good for clean digital pages),
* once on the **adaptive-threshold binary** (good for photographed/scanned pages with uneven lighting).

For each variant we capture:

* `image_to_string` - the raw text string.
* `image_to_data` - per-word bounding boxes, confidence scores and layout indices (block, paragraph, line, word). This metadata is what makes section categorisation, table cell extraction and multi-modal integration possible later in the pipeline.


In [ ]:
def run_ocr(img, psm=6):
    config = f'--oem 1 --psm {psm}'
    text = pytesseract.image_to_string(img, config=config)
    data = pytesseract.image_to_data(img, config=config, output_type=Output.DICT)
    df = pd.DataFrame(data)
    df['conf'] = pd.to_numeric(df['conf'], errors='coerce')
    df = df[(df['conf'] > 0) & df['text'].str.strip().ne('')]
    df = df.reset_index(drop=True)
    return text, df


def best_ocr(stages):
    """Run OCR on the raw grayscale and on the preprocessed binary and keep
    whichever has the higher mean word confidence. Returns the chosen variant
    plus the comparison for reporting."""
    raw_text, raw_df = run_ocr(stages['deskewed_gray'])
    proc_text, proc_df = run_ocr(stages['deskewed_bin'])
    variants = {
        'raw_grayscale': {'text': raw_text, 'df': raw_df,
                          'mean_conf': raw_df['conf'].mean() if len(raw_df) else 0.0},
        'preprocessed':  {'text': proc_text, 'df': proc_df,
                          'mean_conf': proc_df['conf'].mean() if len(proc_df) else 0.0},
    }
    best_name = max(variants, key=lambda k: variants[k]['mean_conf'])
    return best_name, variants


best_name, ocr_demo = best_ocr(stages)
print(f'Variant chosen for {demo_label}: {best_name}')
for name, v in ocr_demo.items():
    marker = '<-- selected' if name == best_name else ''
    print(f'  {name:14s} | {len(v["df"]):>4} words | mean conf {v["mean_conf"]:.1f}  {marker}')

print('\n--- first 400 chars of selected OCR output ---')
print(ocr_demo[best_name]['text'][:400])


In [ ]:
# Compare OCR confidence histograms for the two variants
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (name, v) in zip(axes, ocr_demo.items()):
    ax.hist(v['df']['conf'], bins=20, color='steelblue', edgecolor='black')
    title = f'{name}  ({len(v["df"])} words, mean {v["mean_conf"]:.1f})'
    if name == best_name:
        title += '  <-- selected'
    ax.set_title(title)
    ax.set_xlabel('Confidence (%)')
    ax.set_ylabel('Words')
    ax.axvline(60, color='red', linestyle='--')
plt.suptitle(f'OCR confidence distribution on {demo_label}', fontsize=12)
plt.tight_layout()
plt.show()


# 5. Text preprocessing

Once we have raw text we apply the standard NLP cleaning pipeline taught in Labs 2a (Tokenisation), 2b (Stop Words), 3a (Stemming) and 3b (Lemmatisation):

1. Strip OCR junk - any character outside the printable ASCII range is replaced with a space.
2. Tokenise with **spaCy** (rich token attributes, used downstream by NER and key-phrase extraction).
3. Remove **stop words** with the NLTK English list (extended with a small set of OCR artefacts like single letters).
4. Generate both **Porter stems** and **spaCy lemmas** so we can compare the two reduction strategies.


In [ ]:
EXTRA_STOPWORDS = {'—', '–', "'s", "'re", "'ve"}
ALL_STOPWORDS = ENGLISH_STOPWORDS | EXTRA_STOPWORDS


def clean_raw_text(text):
    text = re.sub(r'[^\x20-\x7E\n]', ' ', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def preprocess_text(raw):
    cleaned = clean_raw_text(raw)
    doc = nlp(cleaned)
    tokens = [t.text for t in doc if not t.is_space and not t.is_punct]
    lowered = [t.lower() for t in tokens if t.strip()]
    filtered = [t for t in lowered if t not in ALL_STOPWORDS and len(t) > 1]
    stems = [PORTER.stem(t) for t in filtered]
    lemmas = [t.lemma_.lower() for t in doc
              if not t.is_space and not t.is_punct
              and t.lemma_.lower() not in ALL_STOPWORDS
              and len(t.lemma_) > 1]
    return {
        'cleaned_text': cleaned,
        'tokens': tokens,
        'no_stop': filtered,
        'stems': stems,
        'lemmas': lemmas,
    }


text_demo = preprocess_text(ocr_demo[best_name]['text'])
print(f"raw tokens : {len(text_demo['tokens']):>4}  | first 10: {text_demo['tokens'][:10]}")
print(f"no-stops   : {len(text_demo['no_stop']):>4}  | first 10: {text_demo['no_stop'][:10]}")
print(f"stems      : {len(text_demo['stems']):>4}  | first 10: {text_demo['stems'][:10]}")
print(f"lemmas     : {len(text_demo['lemmas']):>4}  | first 10: {text_demo['lemmas'][:10]}")


# 6. Text feature extraction

With cleaned text per document we can now extract higher-level features:

* **TF-IDF and Bag-of-Words** over the three documents reveal terms that distinguish each page from the others (Lab 3c).
* **Key phrases** are extracted from spaCy noun chunks - useful for summarising what each section is about.
* **Named entities** are the people, organisations and dates mentioned in the document (Lab 2a - NER).
* **Sentiment** is computed with NLTK's VADER. Document-level polarity isn't very meaningful for academic forms, but it is part of the rubric and gives a sanity check.


In [ ]:
# Run OCR + text preprocessing for every page and cache the results.
# Pick the OCR variant (raw vs preprocessed) with the higher mean confidence per page.
def process_page(sample):
    stages = preprocess_image(sample['bgr'])
    best_name, variants = best_ocr(stages)
    chosen = variants[best_name]
    text_features = preprocess_text(chosen['text'])
    return {
        'source': sample['source'],
        'label': sample['label'],
        'bgr': sample['bgr'],
        'stages': stages,
        'ocr_variant': best_name,
        'ocr_variants': variants,
        'ocr_text': chosen['text'],
        'ocr_df': chosen['df'],
        'mean_conf': chosen['mean_conf'],
        'text': text_features,
    }


DOCS = [process_page(s) for s in SAMPLE_PAGES]
for d in DOCS:
    print(f"{d['label']:30s} | variant={d['ocr_variant']:14s} | "
          f"{len(d['ocr_df']):>4} OCR words | conf {d['mean_conf']:.1f} | "
          f"{len(d['text']['no_stop'])} content tokens")


In [ ]:
# TF-IDF across all three documents
corpus = [' '.join(d['text']['lemmas']) for d in DOCS]
tfidf = TfidfVectorizer(max_features=400, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(corpus)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=[d['label'] for d in DOCS],
)
print('TF-IDF matrix shape:', tfidf_df.shape)

# Top 10 distinguishing terms per document
top_terms = {name: row.sort_values(ascending=False).head(10).round(3)
             for name, row in tfidf_df.iterrows()}
for name, series in top_terms.items():
    print(f"\nTop terms - {name}")
    print(series.to_string())


In [ ]:
# Bag-of-Words for comparison
bow = CountVectorizer(max_features=200, stop_words='english')
bow_matrix = bow.fit_transform(corpus)
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=bow.get_feature_names_out(),
    index=[d['label'] for d in DOCS],
)
print('Top BoW terms (total counts across all docs):')
bow_df.sum().sort_values(ascending=False).head(15)


In [ ]:
# Key phrases (noun chunks) + Named entities + sentiment per document
def text_insights(doc_data):
    doc = nlp(doc_data['text']['cleaned_text'])
    chunks = [c.text.strip() for c in doc.noun_chunks if len(c.text.strip()) > 2]
    chunks = [c for c in chunks if c.lower() not in ALL_STOPWORDS]
    entities = [(e.text.strip(), e.label_) for e in doc.ents if e.text.strip()]
    sentiment = SIA.polarity_scores(doc_data['text']['cleaned_text'])
    return {
        'top_phrases': Counter(chunks).most_common(8),
        'entities': Counter(entities).most_common(10),
        'sentiment': sentiment,
    }


INSIGHTS = [text_insights(d) for d in DOCS]
for d, ins in zip(DOCS, INSIGHTS):
    print(f"\n=== {d['label']} ===")
    print('Top phrases :', ins['top_phrases'][:5])
    print('Entities    :', ins['entities'][:5])
    print('Sentiment   :', ins['sentiment'])


# 7. Text analysis: sections, tables and cross-references

This stage turns the flat OCR output into a structured view of the document:

* **Section categorisation** - Tesseract reports a bounding-box height for every word. Words whose height is well above the document median are flagged as **headings**; smaller words are body text. This is a simple but reliable proxy when no font metadata is available.
* **Tabular data** - we use Tesseract's `block_num`/`line_num` indices to reconstruct lines, then group lines into the table when their bounding boxes line up vertically (covered by the visual table detector in Step 8).
* **Cross-references** - a regex picks up mentions like *Figure 2*, *Table 1*, *Section 4.1*. These references are stored so that we can later link them to the visual region detected by the CV side of the pipeline.


In [ ]:
def categorise_sections(ocr_df):
    """Group OCR words into lines and mark each line as heading or body.

    A line is a heading when its tallest character is at least 1.6x the
    document median *and* at least 6 pixels taller than the median. The
    second condition stops normal ascenders/descenders being mistaken for
    headings on dense pages.
    """
    if ocr_df.empty:
        return pd.DataFrame(columns=['block_num', 'line_num', 'text', 'height', 'role'])
    df = ocr_df.copy()
    median_h = df['height'].median()
    lines = (
        df.groupby(['block_num', 'par_num', 'line_num'])
          .agg(text=('text', lambda x: ' '.join(x)),
               height=('height', 'max'),
               top=('top', 'min'),
               left=('left', 'min'),
               width=('width', 'sum'),
               conf=('conf', 'mean'))
          .reset_index()
    )
    is_heading = (lines['height'] >= 1.6 * median_h) & (lines['height'] >= median_h + 6)
    lines['role'] = np.where(is_heading, 'heading', 'body')
    return lines


# Strong references include a number (e.g. "Figure 2"), generic mentions still
# count for multi-modal binding because they show a *kind* of visual element is
# being discussed.
STRONG_REF_PATTERN = re.compile(
    r'\b(Figure|Fig\.?|Table|Section|Diagram|Chart|Appendix)\s+'
    r'(\d+(?:\.\d+)?[A-Za-z]?)\b',
    flags=re.IGNORECASE,
)
GENERIC_REF_PATTERN = re.compile(
    r'\b(figure|figures|table|tables|diagram|diagrams|chart|charts|signature|signatures|image|images)\b',
    flags=re.IGNORECASE,
)


def find_cross_refs(text):
    refs = []
    seen_strong = set()
    for m in STRONG_REF_PATTERN.finditer(text):
        refs.append({
            'kind': m.group(1).title().rstrip('.'),
            'number': m.group(2),
            'span': m.span(),
            'text': m.group(0),
            'strength': 'strong',
        })
        seen_strong.add(m.span())
    for m in GENERIC_REF_PATTERN.finditer(text):
        if any(s[0] <= m.start() < s[1] for s in seen_strong):
            continue
        refs.append({
            'kind': m.group(1).rstrip('s').title(),
            'number': None,
            'span': m.span(),
            'text': m.group(0),
            'strength': 'generic',
        })
    return refs


sections_demo = categorise_sections(DOCS[1]['ocr_df'])
heading_count = (sections_demo['role'] == 'heading').sum()
print(f"Section categorisation for {DOCS[1]['label']}: "
      f"{heading_count} heading lines / {len(sections_demo)} lines")
print('\nTop 5 tallest lines (likely titles/headings):')
print(sections_demo.sort_values('height', ascending=False)
      [['role', 'height', 'text']].head(5).to_string(index=False))

# Cross-references across all docs - the assignment brief uses generic mentions
for d in DOCS:
    refs_d = find_cross_refs(d['ocr_text'])
    strong = sum(1 for r in refs_d if r['strength'] == 'strong')
    generic = sum(1 for r in refs_d if r['strength'] == 'generic')
    print(f"\n{d['label']}: {strong} strong, {generic} generic")
    for r in refs_d[:5]:
        print(f"  - {r['strength']:7s} {r['text']}")


# 8. Computer-vision feature detection

The CV side of the pipeline detects the **visual** elements of the page:

* **Text blocks** are taken straight from Tesseract's `block_num` groups.
* **Tables** are detected with horizontal and vertical morphological line filters (Lab 6a - *Morphological Operations*) - the intersection of the two masks gives the table grid.
* **Figures / diagrams** are large connected components that contain little or no high-confidence text (Lab 6a - *Connected Components*, Lab 8a - *Contour Detection*).
* **Signatures** would normally be detected as small low-density blobs in the lower-right area; we look for low-solidity contours and flag them if any are found.
* **Edges and contours** are computed with Canny + `findContours` (Lab 8a - *Edge Detection*, *Contour Detection*).


In [ ]:
def detect_text_blocks(ocr_df):
    if ocr_df.empty:
        return []
    df = ocr_df.copy()
    df['right'] = df['left'] + df['width']
    df['bottom'] = df['top'] + df['height']
    blocks = (
        df.groupby('block_num')
          .agg(x=('left', 'min'), y=('top', 'min'),
               x2=('right', 'max'), y2=('bottom', 'max'),
               text=('text', lambda x: ' '.join(x)),
               conf=('conf', 'mean'))
          .reset_index()
    )
    blocks['w'] = blocks['x2'] - blocks['x']
    blocks['h'] = blocks['y2'] - blocks['y']
    blocks = blocks[(blocks['w'] > 30) & (blocks['h'] > 15)]
    return blocks[['x', 'y', 'w', 'h', 'conf', 'text']].to_dict('records')


def detect_tables(binary_img, min_line_len=40):
    inv = 255 - binary_img
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (min_line_len, 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, min_line_len))
    h_lines = cv2.morphologyEx(inv, cv2.MORPH_OPEN, h_kernel)
    v_lines = cv2.morphologyEx(inv, cv2.MORPH_OPEN, v_kernel)
    grid = cv2.add(h_lines, v_lines)
    grid = cv2.dilate(grid, np.ones((3, 3), np.uint8))
    contours, _ = cv2.findContours(grid, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    tables = []
    img_area = binary_img.shape[0] * binary_img.shape[1]
    for c in contours:
        area = cv2.contourArea(c)
        if area < 0.005 * img_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        if w < 80 or h < 40:
            continue
        tables.append({'x': x, 'y': y, 'w': w, 'h': h, 'area': int(area)})
    return tables, grid


def detect_figures(binary_img, text_blocks, min_frac=0.01):
    inv = 255 - binary_img
    closed = cv2.morphologyEx(inv, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25)))
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    figures = []
    img_h, img_w = binary_img.shape
    img_area = img_h * img_w
    text_boxes = [(b['x'], b['y'], b['x'] + b['w'], b['y'] + b['h']) for b in text_blocks]
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_frac * img_area:
            continue
        x, y, w, h = cv2.boundingRect(c)
        cx, cy = x + w // 2, y + h // 2
        if any(tx <= cx <= tx2 and ty <= cy <= ty2 for tx, ty, tx2, ty2 in text_boxes):
            continue
        figures.append({'x': x, 'y': y, 'w': w, 'h': h, 'area': int(area)})
    return figures


def detect_signatures(binary_img):
    inv = 255 - binary_img
    contours, _ = cv2.findContours(inv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    sigs = []
    for c in contours:
        area = cv2.contourArea(c)
        if not 300 < area < 8000:
            continue
        hull = cv2.convexHull(c)
        hull_area = cv2.contourArea(hull)
        if hull_area == 0:
            continue
        solidity = area / hull_area
        x, y, w, h = cv2.boundingRect(c)
        if solidity < 0.35 and w > 40 and h > 15:
            sigs.append({'x': x, 'y': y, 'w': w, 'h': h, 'solidity': round(solidity, 2)})
    return sigs


# Run all detectors on the rubric page (it has the clearest table)
doc1 = DOCS[1]
bin_img = doc1['stages']['deskewed_bin']
text_blocks_1 = detect_text_blocks(doc1['ocr_df'])
tables_1, grid_1 = detect_tables(bin_img)
figures_1 = detect_figures(bin_img, text_blocks_1)
sigs_1 = detect_signatures(bin_img)
edges_1 = cv2.Canny(doc1['stages']['gray'], 80, 200)

print(f'{doc1["label"]}: {len(text_blocks_1)} text blocks | {len(tables_1)} tables | {len(figures_1)} figures | {len(sigs_1)} candidate signatures')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 8))
axes[0].imshow(edges_1, cmap='gray')
axes[0].set_title('Canny edges')
axes[1].imshow(grid_1, cmap='gray')
axes[1].set_title('Table-line mask (H + V morphology)')
annotated = cv2.cvtColor(doc1['bgr'], cv2.COLOR_BGR2RGB).copy()
for b in text_blocks_1:
    cv2.rectangle(annotated, (b['x'], b['y']), (b['x'] + b['w'], b['y'] + b['h']), (0, 200, 0), 2)
for t in tables_1:
    cv2.rectangle(annotated, (t['x'], t['y']), (t['x'] + t['w'], t['y'] + t['h']), (220, 30, 30), 4)
for f in figures_1:
    cv2.rectangle(annotated, (f['x'], f['y']), (f['x'] + f['w'], f['y'] + f['h']), (30, 30, 220), 4)
axes[2].imshow(annotated)
axes[2].set_title('Detected: text (green) | tables (red) | figures (blue)')
for ax in axes:
    ax.axis('off')
plt.suptitle(f'Feature detection - {doc1["label"]}', fontsize=14)
plt.tight_layout()
plt.show()


## 8.1 Extracting structured tabular data

Rubric criterion 4 explicitly asks for **extracting tables** - not just detecting them. The detected table region is cropped, ruled lines are recovered with horizontal and vertical morphological filters, and the rows/columns they define are OCR'd cell-by-cell. The result is a `pandas.DataFrame` that mirrors the visual layout of the table.


In [ ]:
def _find_line_positions(line_mask, axis, threshold_fraction=0.5):
    """Return start-index of every contiguous run of pixels along `axis`
    whose density exceeds `threshold_fraction` of the orthogonal extent."""
    if axis == 'horizontal':
        sums = line_mask.sum(axis=1)
        limit = line_mask.shape[1]
    else:
        sums = line_mask.sum(axis=0)
        limit = line_mask.shape[0]
    threshold = threshold_fraction * limit * 255
    positions = []
    in_line = False
    start = 0
    for i, s in enumerate(sums):
        if s > threshold and not in_line:
            start = i
            in_line = True
        elif s <= threshold and in_line:
            positions.append((start + i) // 2)
            in_line = False
    if in_line:
        positions.append((start + len(sums)) // 2)
    return positions


def extract_table_data(bin_img, gray_img, table_region, min_cell_size=15):
    """OCR every cell of a detected table region. Returns a DataFrame."""
    x, y, w, h = table_region['x'], table_region['y'], table_region['w'], table_region['h']
    crop_bin = bin_img[y:y + h, x:x + w]
    crop_gray = gray_img[y:y + h, x:x + w]

    # Use long line kernels relative to crop size
    h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (max(20, w // 20), 1))
    v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(20, h // 20)))
    h_mask = cv2.morphologyEx(255 - crop_bin, cv2.MORPH_OPEN, h_kernel)
    v_mask = cv2.morphologyEx(255 - crop_bin, cv2.MORPH_OPEN, v_kernel)

    rows = _find_line_positions(h_mask, 'horizontal', threshold_fraction=0.3)
    cols = _find_line_positions(v_mask, 'vertical',   threshold_fraction=0.3)

    # Add the table boundaries if they were not picked up
    if not rows or rows[0] > min_cell_size:
        rows = [0] + rows
    if not rows or h - rows[-1] > min_cell_size:
        rows = rows + [h]
    if not cols or cols[0] > min_cell_size:
        cols = [0] + cols
    if not cols or w - cols[-1] > min_cell_size:
        cols = cols + [w]

    # OCR each cell
    cells = []
    for r0, r1 in zip(rows[:-1], rows[1:]):
        if r1 - r0 < min_cell_size:
            continue
        row_cells = []
        for c0, c1 in zip(cols[:-1], cols[1:]):
            if c1 - c0 < min_cell_size:
                continue
            cell_crop = crop_gray[r0:r1, c0:c1]
            text = pytesseract.image_to_string(cell_crop, config='--oem 1 --psm 6').strip()
            text = re.sub(r'\s+', ' ', text)
            row_cells.append(text)
        if row_cells:
            cells.append(row_cells)

    if not cells:
        return pd.DataFrame(), rows, cols
    max_len = max(len(r) for r in cells)
    padded = [r + [''] * (max_len - len(r)) for r in cells]
    return pd.DataFrame(padded), rows, cols


# Try extracting structured data from the rubric table (clearest grid lines)
table_doc = next((d for d in DOCS if 'rubric' in d['label'].lower()), DOCS[1])
table_tables, _ = detect_tables(table_doc['stages']['deskewed_bin'])
print(f"Document: {table_doc['label']} -> {len(table_tables)} table region(s) detected")

if table_tables:
    biggest = max(table_tables, key=lambda t: t['area'])
    table_df, row_lines, col_lines = extract_table_data(
        table_doc['stages']['deskewed_bin'],
        table_doc['stages']['gray'],
        biggest,
    )
    print(f'Recovered grid: {len(row_lines) - 1} rows x {len(col_lines) - 1} columns')
    print(f'OCR cells: {table_df.shape[0]} rows x {table_df.shape[1]} columns')
    # Show the first few rows truncated for readability
    preview = table_df.copy()
    for c in preview.columns:
        preview[c] = preview[c].str.slice(0, 60)
    display(preview.head(8))
else:
    table_df = pd.DataFrame()
    print('No table region available for structured extraction')


## 8.2 Feature matching with ORB

The brief explicitly lists *feature matching (e.g., SIFT, ORB)* as one of the techniques. ORB (Oriented FAST and Rotated BRIEF, Lab 8a) is the free, modern alternative to SIFT and is bundled with OpenCV. Here we use it for a practical document-understanding task: confirming that two pages came from the **same document** by matching repeated visual elements (the ATU header / institutional logo).

For two pages of the same source we expect:

* a dense cluster of matches across the **header region** (the logo and contact block are identical between pages),
* sparse matches across the body (the text is different).


In [ ]:
def orb_match(img1_bgr, img2_bgr, max_matches=50):
    """Detect ORB keypoints in two images and return the best Brute-Force
    Hamming matches plus the visualisation."""
    gray1 = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=2000, scaleFactor=1.2, nlevels=8)
    kp1, des1 = orb.detectAndCompute(gray1, None)
    kp2, des2 = orb.detectAndCompute(gray2, None)
    if des1 is None or des2 is None:
        return None, [], kp1 or [], kp2 or []
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = sorted(bf.match(des1, des2), key=lambda m: m.distance)[:max_matches]
    vis = cv2.drawMatches(
        img1_bgr, kp1, img2_bgr, kp2, matches, None,
        matchColor=(0, 200, 0), singlePointColor=(60, 60, 60),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    return vis, matches, kp1, kp2


# Find two pages from the same PDF source - they share an institutional header
same_source_pages = {}
for d in DOCS:
    same_source_pages.setdefault(d['source'], []).append(d)
matched_pair = next((pages for pages in same_source_pages.values() if len(pages) >= 2), None)

if matched_pair is None:
    print('No multi-page PDF in samples - falling back to matching the first two docs')
    matched_pair = DOCS[:2]

img_a, img_b = matched_pair[0]['bgr'], matched_pair[1]['bgr']
vis, matches, kp1, kp2 = orb_match(img_a, img_b, max_matches=50)
print(f'ORB matching: {matched_pair[0]["label"]} <-> {matched_pair[1]["label"]}')
print(f'Keypoints: page A = {len(kp1)}, page B = {len(kp2)}')
print(f'Top-50 match average Hamming distance: {np.mean([m.distance for m in matches]):.1f}')

plt.figure(figsize=(20, 10))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'ORB feature matching: top 50 matches ({matched_pair[0]["label"]} vs {matched_pair[1]["label"]})')
plt.show()


# 9. Image segmentation

Segmentation is the per-region equivalent of feature detection - here every pixel ends up belonging to a meaningful region. Two techniques are demonstrated:

* **Connected components** on the inverted binary mask groups every contiguous run of dark pixels into a labelled blob. This is the simplest and most reliable way to find character / word / paragraph regions on a document page (Lab 8a - *Image Segmentation*).
* **Watershed** treats the page as a topographic surface and floods it from markers. We use it to separate touching paragraphs/figures. The labelled output is rendered with a colormap so that adjacent regions are visually distinct.


In [ ]:
def segment_connected_components(binary_img, min_area=80):
    inv = 255 - binary_img
    n, labels, stats, centroids = cv2.connectedComponentsWithStats(inv, connectivity=8)
    keep = stats[:, cv2.CC_STAT_AREA] >= min_area
    keep[0] = False  # drop the background label
    kept_labels = np.where(keep)[0]
    mask = np.isin(labels, kept_labels)
    coloured = np.zeros((*labels.shape, 3), dtype=np.uint8)
    rng = np.random.default_rng(0)
    palette = rng.integers(40, 255, size=(n, 3))
    palette[0] = (0, 0, 0)
    for lbl in kept_labels:
        coloured[labels == lbl] = palette[lbl]
    return n, labels, stats, coloured, mask


def segment_watershed(binary_img):
    inv = 255 - binary_img
    dist = cv2.distanceTransform(inv, cv2.DIST_L2, 5)
    _, fg = cv2.threshold(dist, 0.4 * dist.max(), 255, 0)
    fg = fg.astype(np.uint8)
    bg = cv2.dilate(inv, np.ones((5, 5), np.uint8), iterations=2)
    unknown = cv2.subtract(bg, fg)
    n_markers, markers = cv2.connectedComponents(fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    coloured = cv2.cvtColor(binary_img, cv2.COLOR_GRAY2BGR)
    markers = cv2.watershed(coloured, markers)
    cmap = plt.get_cmap('tab20b')
    out = np.zeros((*markers.shape, 3), dtype=np.uint8)
    for lbl in np.unique(markers):
        if lbl <= 0:
            continue
        colour = (np.array(cmap(lbl % 20))[:3] * 255).astype(np.uint8)
        out[markers == lbl] = colour
    return markers, out


n_cc, _, stats_cc, cc_vis, _ = segment_connected_components(bin_img)
_, ws_vis = segment_watershed(bin_img)
print(f'Connected components: {n_cc - 1} kept regions')

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(cc_vis)
axes[0].set_title('Connected-component segmentation')
axes[1].imshow(ws_vis)
axes[1].set_title('Watershed segmentation')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


# 10. Multi-modal integration

The previous stages give us:

* `cross_refs` - mentions of tables/figures found **in the text**.
* `tables` / `figures` / `text_blocks` - regions found **in the image**.

Integrating them means assigning every textual reference to a candidate visual region. We do that by counting references per kind and pairing them with detected regions of the same kind in reading order (top-to-bottom). If there are more references than regions, or vice versa, the surplus is recorded as unresolved.

The annotated output shows every detected region, labelled with its type and (where available) the textual reference it satisfies.


In [ ]:
def integrate(doc):
    bin_img = doc['stages']['deskewed_bin']
    text_blocks = detect_text_blocks(doc['ocr_df'])
    tables, _ = detect_tables(bin_img)
    figures = detect_figures(bin_img, text_blocks)
    refs = find_cross_refs(doc['ocr_text'])
    sections = categorise_sections(doc['ocr_df'])

    # Pair references with regions of the same kind, in reading order.
    # Strong references (with a number) bind first, then generic mentions.
    refs_by_kind = {'table': [], 'figure': []}
    for r in refs:
        kind_key = r['kind'].lower()
        if kind_key in ('fig', 'diagram', 'chart', 'image'):
            kind_key = 'figure'
        if kind_key in refs_by_kind:
            refs_by_kind[kind_key].append(r)

    def assign(region_list, kind_key):
        sorted_regions = sorted(region_list, key=lambda r: (r['y'], r['x']))
        # Strong references first, then generic
        refs_sorted = sorted(refs_by_kind.get(kind_key, []),
                             key=lambda r: 0 if r['strength'] == 'strong' else 1)
        for i, region in enumerate(sorted_regions):
            if i < len(refs_sorted):
                region['reference'] = refs_sorted[i]['text']
                region['reference_strength'] = refs_sorted[i]['strength']
            else:
                region['reference'] = None
                region['reference_strength'] = None
        return sorted_regions

    tables = assign(tables, 'table')
    figures = assign(figures, 'figure')

    return {
        'text_blocks': text_blocks,
        'tables': tables,
        'figures': figures,
        'cross_refs': refs,
        'sections': sections,
    }


def annotate(doc, integration):
    img = cv2.cvtColor(doc['bgr'], cv2.COLOR_BGR2RGB).copy()
    for b in integration['text_blocks']:
        cv2.rectangle(img, (b['x'], b['y']), (b['x'] + b['w'], b['y'] + b['h']), (0, 180, 0), 2)
    for t in integration['tables']:
        cv2.rectangle(img, (t['x'], t['y']), (t['x'] + t['w'], t['y'] + t['h']), (220, 30, 30), 4)
        label = f"Table ({t['reference']})" if t['reference'] else 'Table'
        cv2.putText(img, label, (t['x'], max(t['y'] - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (220, 30, 30), 2)
    for f in integration['figures']:
        cv2.rectangle(img, (f['x'], f['y']), (f['x'] + f['w'], f['y'] + f['h']), (30, 30, 220), 4)
        label = f"Figure ({f['reference']})" if f['reference'] else 'Figure'
        cv2.putText(img, label, (f['x'], max(f['y'] - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (30, 30, 220), 2)
    # Headings overlay
    for _, row in integration['sections'].iterrows():
        if row['role'] == 'heading':
            x, y, w, h = int(row['left']), int(row['top']), int(row['width']), int(row['height'])
            cv2.rectangle(img, (x, y), (x + w, y + h), (200, 150, 0), 2)
    return img


INTEGRATION = [integrate(d) for d in DOCS]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 9))
for ax, doc, integ in zip(axes, DOCS, INTEGRATION):
    ax.imshow(annotate(doc, integ))
    ax.set_title(f"{doc['label']}\n"
                 f"{len(integ['text_blocks'])} text | {len(integ['tables'])} tables | "
                 f"{len(integ['figures'])} figures | {len(integ['cross_refs'])} refs",
                 fontsize=11)
    ax.axis('off')
plt.suptitle('Multi-modal annotation: text blocks (green), headings (orange), tables (red), figures (blue)',
             fontsize=13)
plt.tight_layout()
plt.show()


# 11. Final structured report

The final output combines everything produced by the pipeline into a single artefact per document:

* **Cleaned text** - the lemmatised, stop-word-stripped version of the OCR output.
* **Insights** - top TF-IDF terms, top noun-chunk key phrases, named entities, sentiment scores.
* **Structured layout** - number of text blocks, tables, figures, headings and cross-references with their resolved bindings.
* **Annotated image** - the page with every detected region rendered on top.

A `DataFrame` consolidates the per-document summary so the result can be exported, persisted or fed into a downstream system.


In [ ]:
def build_report(doc, integ, insights, tfidf_row):
    cleaned = doc['text']['cleaned_text']
    top_tfidf = tfidf_row.sort_values(ascending=False).head(8)
    # Show the tallest headings first - they are the most likely real titles
    heading_rows = (integ['sections'][integ['sections']['role'] == 'heading']
                    .sort_values('height', ascending=False))
    headings = heading_rows['text'].tolist()
    refs_resolved = sum(1 for t in integ['tables'] + integ['figures'] if t.get('reference'))
    return {
        'document': doc['label'],
        'words_ocr': len(doc['ocr_df']),
        'mean_conf': round(doc['ocr_df']['conf'].mean(), 1),
        'content_tokens': len(doc['text']['no_stop']),
        'headings': len(headings),
        'text_blocks': len(integ['text_blocks']),
        'tables': len(integ['tables']),
        'figures': len(integ['figures']),
        'cross_refs': len(integ['cross_refs']),
        'refs_resolved': refs_resolved,
        'sentiment': insights['sentiment']['compound'],
        'top_terms': ', '.join(top_tfidf.index[:5]),
        'top_entities': ', '.join(f"{e[0][0]}({e[0][1]})" for e in insights['entities'][:5]),
        'top_phrases': ', '.join(p[0] for p in insights['top_phrases'][:5]),
        'preview_text': (cleaned[:200] + '...') if len(cleaned) > 200 else cleaned,
        'heading_examples': headings[:3],
    }


report_rows = [
    build_report(d, integ, ins, tfidf_df.loc[d['label']])
    for d, integ, ins in zip(DOCS, INTEGRATION, INSIGHTS)
]
report_df = pd.DataFrame(report_rows)

pd.set_option('display.max_colwidth', 80)
report_df[['document', 'words_ocr', 'mean_conf', 'content_tokens',
           'headings', 'text_blocks', 'tables', 'figures', 'cross_refs',
           'refs_resolved', 'sentiment', 'top_terms']]


In [ ]:
# Print a per-document narrative summary
for row in report_rows:
    print('=' * 70)
    print(f"Document : {row['document']}")
    print(f"OCR      : {row['words_ocr']} words, mean confidence {row['mean_conf']}%")
    print(f"Layout   : {row['text_blocks']} text blocks | {row['headings']} headings | "
          f"{row['tables']} tables | {row['figures']} figures")
    print(f"Refs     : {row['cross_refs']} found in text, {row['refs_resolved']} bound to a visual region")
    print(f"Sentiment: compound={row['sentiment']}")
    print(f"Top TF-IDF terms : {row['top_terms']}")
    print(f"Top entities     : {row['top_entities']}")
    print(f"Top key phrases  : {row['top_phrases']}")
    print(f"Heading examples : {row['heading_examples']}")
    print(f"Text preview     : {row['preview_text']}")
print('=' * 70)


In [ ]:
# One final side-by-side: original page next to fully annotated version
for doc, integ in zip(DOCS, INTEGRATION):
    fig, axes = plt.subplots(1, 2, figsize=(16, 9))
    axes[0].imshow(cv2.cvtColor(doc['bgr'], cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original - {doc['label']}")
    axes[1].imshow(annotate(doc, integ))
    axes[1].set_title('Annotated (text blocks, headings, tables, figures)')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## Challenges encountered

* **OCR on photographs vs scans** - photographs of pages exhibit uneven lighting, so a single global threshold (Otsu) produced large black patches. Switching to adaptive Gaussian thresholding with a large block size (31) resolved this; CLAHE enhancement is applied first so the threshold has well-spread tones to work with.
* **Choosing per-document preprocessing** - aggressive binarisation hurt the clean digital page (`TextCleansing.jpg`) but helped the scanned-style pages. The pipeline therefore runs OCR on **both** the raw grayscale and the preprocessed binary and keeps whichever variant has the higher mean confidence.
* **Skew estimation bug** - `np.where` returns coordinates in `(row, col)` but `cv2.minAreaRect` expects `(x, y)`. Forgetting the swap produced a ~90deg "skew" estimate and destroyed OCR on aligned pages. The fix is documented inline in `estimate_skew_angle` so the gotcha is not silently re-introduced.
* **Headings vs body separation** - PDFs do not embed font information once rasterised. The per-word height heuristic (>=1.6x median AND >=median+6 px) turned out to be more reliable than chasing OCR `font_size`, which Tesseract does not populate consistently.
* **Distinguishing tables from dense paragraphs** - early experiments using contour area alone produced many false positives. The horizontal + vertical morphological filter only fires on actual ruled lines, eliminating false positives entirely. The same filter is reused inside `extract_table_data` to recover row/column boundaries when OCR'ing cells.
* **Multi-modal binding** - matching textual references to visual regions purely by counting can mis-pair them. The pipeline pairs strong "Figure N"/"Table N" references first, falls back to generic mentions, and records any unresolved references in the final report so the limitation is visible rather than hidden.
* **ORB descriptor collapse** - on PDF pages rendered from the *same* source, the header keypoints are pixel-identical and produce Hamming distance 0. The match list still works as a similarity signal but the distance threshold is uninformative; the demo prints both the match count and the mean distance so the reader can interpret it.

## Mapping to the assignment brief

| Brief requirement | Notebook section |
| ----------------- | ---------------- |
| Tesseract OCR on diverse document types (PDFs included) | s2, s4 (PDFs ingested directly via `load_document_pages`) |
| Tokenisation, stemming, lemmatisation, stopword removal | s5 |
| TF-IDF / BoW, key phrases, NER, sentiment | s6 |
| Section categorisation, tabular data extraction, text-image relationships | s7, s8.1 |
| OpenCV preprocessing (grayscale, threshold, noise reduction) | s3 |
| Edge detection, feature matching (SIFT/ORB), contour detection | s8, s8.2 |
| Watershed / connected-components segmentation | s9 |
| Image enhancement (contrast adjustment, deblurring, binarisation) | s3 (CLAHE + unsharp mask + adaptive threshold) |
| Text + image integration with linked references | s10 |
| Final output: cleaned text, refined image, insights and reports | s11 |

## References

* SpaCy documentation - tokenisation, NER, lemmatisation: https://spacy.io/api
* NLTK book - stop words, stemming, VADER sentiment: https://www.nltk.org/book/
* Tesseract page-segmentation modes and TSV output: https://tesseract-ocr.github.io/tessdoc/
* OpenCV docs - adaptive thresholding, morphology, watershed, ORB, CLAHE: https://docs.opencv.org/4.x/
* scikit-learn TfidfVectorizer / CountVectorizer: https://scikit-learn.org/stable/modules/feature_extraction.html
* PyMuPDF (`pymupdf`) - rendering PDF pages to images: https://pymupdf.readthedocs.io/
* Skew estimation via `cv2.minAreaRect` - PyImageSearch / OpenCV community
* Pipeline ideas adapted from Labs 1c (PDF), 2a/2b (tokenisation, stop words), 3a/3b/3c (stemming, lemmatisation, TF-IDF), 6a (image processing - thresholding, morphology, blur), 8a (edge detection, contours, ORB/SIFT, watershed) and `Lab_Tesseract_OCR` (Tesseract OEM/PSM, `image_to_data`).
* Cloude Code used to document sections of the code and help with refactoring